# Intent and Entity Extraction With Ground Truth

This notebook evaluates prompt-based extraction of `priority` and `type` from customer support tickets using a labeled dataset.

In [66]:
from datasets import load_dataset
MAX_ROWS = 2000  # lower if still slow
ds = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")

## Dataset

We use the `Tobi-Bueck/customer-support-tickets` dataset. Each row contains a ticket subject, a ticket body, and ground-truth labels for `priority` and `type`.

Example:

```
priority: high | type: Incident
Subject: None
Body: Customer Support reports intermittent service outages affecting multiple users. Recent deployment appears to have caused connectivity issues. Restarting the server and applying updates resolved the problem. Further investigation is required to identify the root cause and implement a solution.
```

## Label Space

`priority`: `very_low`, `low`, `medium`, `high`, `critical`

`type`: `Incident`, `Request`, `Problem`, `Change`

## Objective

Compare three prompting strategies for extracting `priority` and `type` from the ticket text:

1. Direct extraction
2. Self-review in one prompt
3. Separate review prompt

Because the dataset includes ground truth, we can compare the exact-match percentage across all approaches.

## Setup

Load the libraries required for prompt rendering, model calls, and evaluation.

In [67]:
import json
import jinja2
from openai import OpenAI
import pandas as pd

## Sample A Labeled Subset and Few-Shot Examples

Select a small batch of labeled tickets to keep the experiment fast and comparable across prompting strategies.
This subset will be used for the main evaluation. We also create a separate few-shot set to provide examples in the prompts without data leakage.

In [68]:
# Fast, leakage-safe sampling (no expensive "x not in sampled_ds")
n_samples = 20
n_few_shots = 5

ALLOWED_PRIORITY = {"very_low", "low", "medium", "high", "critical"}
ALLOWED_TYPE = {"Incident", "Request", "Problem", "Change"}

labeled_ds = ds.filter(
    lambda x: x["priority"] in ALLOWED_PRIORITY and x["type"] in ALLOWED_TYPE
)

shuffled = labeled_ds.shuffle(seed=42)
k_eval = min(n_samples, len(shuffled))
k_few = min(n_few_shots, max(0, len(shuffled) - k_eval))

sampled_ds = shuffled.select(range(0, k_eval))
few_shot_ds = shuffled.select(range(k_eval, k_eval + k_few))

# Print one example to verify
print("Sampled")
print(json.dumps(sampled_ds[0], indent=2))  
print("\nFew-Shot")
print(json.dumps(few_shot_ds[0], indent=2))


Sampled
{
  "subject": "Assistance Needed: Issue with Data Encryption",
  "body": "I am writing to report an unexpected failure with the data encryption, which might be due to a recent software update. So far, I have attempted to restart my system and reviewed the configurations, but the issue persists. I would greatly appreciate your assistance in resolving this problem or providing guidance on how to fix it. Please inform me of any additional steps I need to take or if you require further information to better understand and address the issue. I look forward to hearing from you and finding a solution. Thank you for your time and support.",
  "answer": "I have received your email regarding the unexpected failure with the data encryption. To better assist you, could you please provide me with more details about the software update and the error message you received? Additionally, I will need access to your customer information to further investigate the issue. If convenient, I can also

## Prompt Design

In [69]:
prompt_env = jinja2.Environment(
    loader=jinja2.FileSystemLoader("prompts"),
    autoescape=False,
)

# Load system and user templates for all strategies
no_review_system_template = prompt_env.get_template("no_review_system_prompt.j2")
selfreview_system_template = prompt_env.get_template("selfreview_system_prompt.j2")
review_system_template = prompt_env.get_template("review_system_prompt.j2")

no_review_user_template = prompt_env.get_template("no_review_user_prompt.j2")
selfreview_user_template = prompt_env.get_template("selfreview_user_prompt.j2")
review_user_template = prompt_env.get_template("review_user_prompt.j2")

# Pre-render system prompts (static, no variables)
NO_REVIEW_SYSTEM_PROMPT = no_review_system_template.render()
SELFREVIEW_SYSTEM_PROMPT = selfreview_system_template.render()
REVIEW_SYSTEM_PROMPT = review_system_template.render()

subject_col = "subject"
description_col = "body"
priority_col = "priority"
incident_col = "type"

# Formatted few-shot examples for in-context learning, using original text and JSON labels for clarity and consistency with templates
few_shots_text = "\n\n".join(
    f"Subject: {row.get(subject_col)}\n"
    f"Description: {row.get(description_col, '')}\n"
    f"Output: {json.dumps({'priority': row[priority_col], 'type': row[incident_col]})}"
    for row in few_shot_ds
)

# Keep template input fields separate from evaluation labels
template_inputs = [
    {
        "subject": row.get(subject_col),
        "description": row.get(description_col, ""),
    }
    for row in sampled_ds
]

eval_labels = [
    {
        "priority": row[priority_col],
        "type": row[incident_col],
    }
    for row in sampled_ds
]

# Combined structure used by downstream evaluation cells
sample_rows = [
    {**inp, **lbl}
    for inp, lbl in zip(template_inputs, eval_labels)
]

def render_user_prompts(template, contexts):
    return [template.render(**ctx) for ctx in contexts]

base_user_contexts = [
    {**inp, "few_shots": few_shots_text}
    for inp in template_inputs
]

filled_no_review_prompts = render_user_prompts(
    no_review_user_template,
    base_user_contexts,
)
filled_selfreview_prompts = render_user_prompts(
    selfreview_user_template,
    base_user_contexts,
)

no_review_messages = [
    {"system": NO_REVIEW_SYSTEM_PROMPT, "user": prompt}
    for prompt in filled_no_review_prompts
]

self_review_messages = [
    {"system": SELFREVIEW_SYSTEM_PROMPT, "user": prompt}
    for prompt in filled_selfreview_prompts
]

# Multiple call prompt review will be filled only after

# Print example prompts to verify
print("No-Review User Prompt Example:")
print(filled_no_review_prompts[0])
print("\nSelf-Review User Prompt Example:")
print(filled_selfreview_prompts[0])

No-Review User Prompt Example:

Few-shot examples:
Subject: Abrechnungsinformationen
Description: Interessiert an unserer SaaS-Projektmanagement-Software und den verfügbaren Zahlungsplänen. Könnten Sie eine detaillierte Aufstellung der Kosten sowie mögliche Rabatte bereitstellen? Zusätzlich wären Informationen zu kostenlosen Testversionen und Demos hilfreich, um eine fundierte Entscheidung darüber treffen zu können, ob diese Lösung den Bedürfnissen entspricht.
Output: {"priority": "medium", "type": "Request"}

Subject: Concern about Database Integration
Description: A financial company is facing issues with integrating with Oracle Database 19c, specifically with the analysis tools, which is hindering efficient data processing. This might be due to outdated software or configuration errors. Despite attempts to update the software and check connections, the problem remains unresolved, causing significant delays and impacting data processing efficiency. We would greatly appreciate your as

## Approach 1: Direct Extraction

Start with a single prompt that predicts `priority` and `type` directly from each ticket.

### Run the model

Call the model on the direct-extraction prompts.

In [70]:
# Function to call the LLM with system + user messages
def call_llm(message_pairs, model="gpt-4.1-mini"):
    client = OpenAI()
    responses = []
    for msg in message_pairs:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": msg["system"]},
                {"role": "user", "content": msg["user"]},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        responses.append(response.choices[0].message.content)
    return responses

In [71]:
no_review_responses = call_llm(no_review_messages)

# Print one response to verify
print("No-Review Response Example:")
print(no_review_responses[0])

No-Review Response Example:
{"priority": "high", "type": "Incident"}


### Evaluate direct extraction

Count a prediction as correct only when both labels match the ground truth.

In [72]:
def parse_response(text):
    text = text.strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        data = json.loads(text[start:end + 1]) if start != -1 and end != -1 else {}
    return {
        "pred_priority": data.get("priority") or data.get("corrected_priority"),
        "pred_type": data.get("type") or data.get("corrected_type"),
    }

def exact_match_percentage(samples, responses):
    correct = sum(
        parse_response(response) == {"pred_priority": sample["priority"], "pred_type": sample["type"]}
        for sample, response in zip(samples, responses)
    )
    return (correct / len(samples) * 100) if samples else 0.0

no_review_accuracy = exact_match_percentage(sample_rows, no_review_responses)
print(f"Direct extraction exact-match: {no_review_accuracy:.2f}%")

example_idx = 1
example_pred = parse_response(no_review_responses[example_idx])
example_gt = {"pred_priority": sample_rows[example_idx]["priority"], "pred_type": sample_rows[example_idx]["type"]}
print(f"Example #{example_idx}")
print(f"Ground truth: {example_gt}")
print(f"Prediction  : {example_pred}")

# Comparison based only on the type
type_match = sum(
    parse_response(response)["pred_type"] == sample["type"]
    for sample, response in zip(sample_rows, no_review_responses)
)
type_accuracy = (type_match / len(sample_rows) * 100) if sample_rows else 0.0
print(f"Type-only exact-match: {type_accuracy:.2f}%")

Direct extraction exact-match: 20.00%
Example #1
Ground truth: {'pred_priority': 'high', 'pred_type': 'Request'}
Prediction  : {'pred_priority': 'medium', 'pred_type': 'Request'}
Type-only exact-match: 55.00%


## Approach 2: Self-Review In One Prompt

Ask the model to classify and self-check in the same prompt before returning final JSON.

In [73]:
self_review_messages = [
    {"system": SELFREVIEW_SYSTEM_PROMPT, "user": prompt}
    for prompt in filled_selfreview_prompts
]

self_review_responses = call_llm(self_review_messages)
self_review_accuracy = exact_match_percentage(sample_rows, self_review_responses)
print(f"Self-review exact-match: {self_review_accuracy:.2f}%")
type_match_self_review = sum(
    parse_response(response)["pred_type"] == sample["type"]
    for sample, response in zip(sample_rows, self_review_responses)
)
type_accuracy_self_review = (type_match_self_review / len(sample_rows) * 100) if sample_rows else 0.0
print(f"Self-review type-only exact-match: {type_accuracy_self_review:.2f}%")

example_idx = 0
example_pred = parse_response(self_review_responses[example_idx])
example_gt = {"pred_priority": sample_rows[example_idx]["priority"], "pred_type": sample_rows[example_idx]["type"]}
print(f"Example #{example_idx}")
print(f"Ground truth: {example_gt}")
print(f"Prediction  : {example_pred}")

# Print review and correction made on the example
print("\nReview and Correction:")
print(self_review_responses[example_idx])

Self-review exact-match: 20.00%
Self-review type-only exact-match: 65.00%
Example #0
Ground truth: {'pred_priority': 'low', 'pred_type': 'Incident'}
Prediction  : {'pred_priority': 'high', 'pred_type': 'Incident'}

Review and Correction:
{
  "priority": "high",
  "type": "Incident",
  "self_review": "The ticket describes an active failure in data encryption causing disruption, which fits the Incident type. The priority is set to high due to the critical nature of encryption issues, though no explicit urgency was stated, high is appropriate given the impact.",
  "corrected_priority": "high",
  "corrected_type": "Incident"
}


## Approach 3: Separate Review Prompt

Generate a first-pass extraction, then send it to a second prompt that critiques and corrects it.

In [74]:
# Reuse first-pass predictions from direct extraction, then apply a separate review step
first_pass_clean = [
    json.dumps(
        {
            "priority": parse_response(response)["pred_priority"],
            "type": parse_response(response)["pred_type"],
        }
    )
    for response in no_review_responses
]

review_contexts = [
    {
        **inp,
        "extraction": clean_response,
        "few_shots": few_shots_text,
    }
    for inp, clean_response in zip(template_inputs, first_pass_clean)
]

review_prompts = render_user_prompts(review_user_template, review_contexts)
review_messages = [
    {"system": REVIEW_SYSTEM_PROMPT, "user": prompt}
    for prompt in review_prompts
]

review_responses = call_llm(review_messages)
separate_review_accuracy = exact_match_percentage(sample_rows, review_responses)
print(f"Separate review exact-match: {separate_review_accuracy:.2f}%")
type_match_review = sum(
    parse_response(response)["pred_type"] == sample["type"]
    for sample, response in zip(sample_rows, review_responses)
)
type_accuracy_review = (type_match_review / len(sample_rows) * 100) if sample_rows else 0.0
print(f"Separate review type-only exact-match: {type_accuracy_review:.2f}%")

example_idx = 0
example_pred = parse_response(review_responses[example_idx])
example_gt = {"pred_priority": sample_rows[example_idx]["priority"], "pred_type": sample_rows[example_idx]["type"]}
print(f"Example #{example_idx}")
print(f"Ground truth: {example_gt}")
print(f"Prediction  : {example_pred}")

Separate review exact-match: 15.00%
Separate review type-only exact-match: 55.00%
Example #0
Ground truth: {'pred_priority': 'low', 'pred_type': 'Incident'}
Prediction  : {'pred_priority': 'medium', 'pred_type': 'Incident'}


## Comparison

Compare the exact-match percentage across all three prompting strategies.

In [75]:
import pandas as pd

comparison = pd.DataFrame(
    [
        {"approach": "Direct extraction", "percentage_correct": no_review_accuracy},
        {"approach": "Self-review in one prompt", "percentage_correct": self_review_accuracy},
        {"approach": "Separate review prompt", "percentage_correct": separate_review_accuracy},
    ]
).sort_values("percentage_correct", ascending=False).reset_index(drop=True)

display(comparison)

,approach,percentage_correct
0,Direct extraction,20.0
1,Self-review in one prompt,20.0
2,Separate review prompt,15.0


## Consideration

As we can see from the results, it's difficult to evaluate directly using yes or no answer.

In fact, the priority can be relative and not absolute, making very difficult to evaluate the responses.

